# Plot synthetische Szenarien (normalisiert) → PDF für LaTeX

Dieses Notebook lädt die drei Scenario-Parquets (MultiIndex: date, asset),
plottet pro Szenario alle Assets normalisiert (Start=1) und speichert PDFs
in deinen LaTeX-Figures-Ordner.


In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# --- Pfade anpassen ---
REPO_ROOT = Path(r"C:/Dev/Bachelorarbeit")
LATEX_FIG_DIR = Path(r"C:/Users/Dr. LongRodeo/Documents/Studium/8. Semester Bacherlorarbeit/latex-ba/figures/scenarios")

SCENARIOS = {
    "bear_1y": REPO_ROOT / "data/synth/bear_1y_features.parquet",
    "side_lowvol_1y": REPO_ROOT / "data/synth/side_lowvol_1y_features.parquet",
    "side_highvol_1y": REPO_ROOT / "data/synth/side_highvol_1y_features.parquet",
}

LATEX_FIG_DIR.mkdir(parents=True, exist_ok=True)
print("Saving to:", LATEX_FIG_DIR)


Saving to: C:\Users\Dr. LongRodeo\Documents\Studium\8. Semester Bacherlorarbeit\latex-ba\figures\scenarios


In [2]:
def load_panel(path: Path) -> pd.DataFrame:
    """Lädt MultiIndex-Panel (date, asset). Nutzt falls vorhanden dein utils.parquet_io."""
    try:
        import sys
        sys.path.insert(0, str(REPO_ROOT / "src"))
        from utils.parquet_io import load_parquet  # type: ignore
        df = load_parquet(path)
    except Exception:
        df = pd.read_parquet(path)
    return df

def panel_to_wide(panel: pd.DataFrame, price_col: str | None = None) -> pd.DataFrame:
    """Gibt Wide-DF zurück: index=date, columns=asset, values=price."""
    if not isinstance(panel.index, pd.MultiIndex):
        raise ValueError("Erwarte MultiIndex (date, asset).")

    cols = list(panel.columns)
    candidates = [price_col] if price_col else []
    candidates += ["adj_close_raw", "close", "price", "px_close"]
    col = next((c for c in candidates if c in cols), None)
    if col is None:
        raise ValueError(f"Kein Preis-Column gefunden. Vorhanden: {cols[:20]}")

    wide = panel[col].unstack("asset").sort_index()
    wide.index = pd.to_datetime(wide.index).normalize()
    return wide

def normalize_start_one(wide: pd.DataFrame) -> pd.DataFrame:
    first = wide.iloc[0]
    return wide.divide(first, axis=1)


In [3]:
TITLE_MAP = {
    "bear_1y": "Einjähriger Bärenmarkt",
    "side_lowvol_1y": "Einjähriger Seitwärtsmarkt (niedrige Volatilität)",
    "side_highvol_1y": "Einjähriger Seitwärtsmarkt (hohe Volatilität)",
}

def plot_scenario(name: str, wide_norm: pd.DataFrame, out_dir: Path):
    plt.figure(figsize=(12, 6))
    for c in wide_norm.columns:
        plt.plot(wide_norm.index, wide_norm[c], label=str(c), linewidth=1.2)
    title = TITLE_MAP.get(name, name)
    plt.title(f"{title} (normalisiert, Start=1)")
    plt.xlabel("Datum")
    plt.ylabel("Preis (Start=1)")
    plt.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
    plt.tight_layout()

    out_path = out_dir / f"synth_norm_{name}.pdf"
    plt.savefig(out_path, bbox_inches="tight")
    plt.close()
    return out_path

saved = []
for name, path in SCENARIOS.items():
    panel = load_panel(path)
    wide = panel_to_wide(panel)
    wide = wide.ffill().bfill()  # robust gegen einzelne Lücken
    wide_norm = normalize_start_one(wide)
    out = plot_scenario(name, wide_norm, LATEX_FIG_DIR)
    saved.append(out)
    print("Saved:", out)

saved


Saved: C:\Users\Dr. LongRodeo\Documents\Studium\8. Semester Bacherlorarbeit\latex-ba\figures\scenarios\synth_norm_bear_1y.pdf
Saved: C:\Users\Dr. LongRodeo\Documents\Studium\8. Semester Bacherlorarbeit\latex-ba\figures\scenarios\synth_norm_side_lowvol_1y.pdf
Saved: C:\Users\Dr. LongRodeo\Documents\Studium\8. Semester Bacherlorarbeit\latex-ba\figures\scenarios\synth_norm_side_highvol_1y.pdf


[WindowsPath('C:/Users/Dr. LongRodeo/Documents/Studium/8. Semester Bacherlorarbeit/latex-ba/figures/scenarios/synth_norm_bear_1y.pdf'),
 WindowsPath('C:/Users/Dr. LongRodeo/Documents/Studium/8. Semester Bacherlorarbeit/latex-ba/figures/scenarios/synth_norm_side_lowvol_1y.pdf'),
 WindowsPath('C:/Users/Dr. LongRodeo/Documents/Studium/8. Semester Bacherlorarbeit/latex-ba/figures/scenarios/synth_norm_side_highvol_1y.pdf')]

## Optional: Mini-Statistik pro Szenario (Return/Vol/MDD pro Asset)
Kannst du nutzen, um im Text 1–2 Sätze zur Regime-Charakteristik zu schreiben.

In [4]:
import numpy as np

def mdd(x: pd.Series) -> float:
    roll_max = x.cummax()
    dd = x / roll_max - 1.0
    return float(dd.min())

def scenario_stats(wide: pd.DataFrame) -> pd.DataFrame:
    r = wide.pct_change().dropna(how="all")
    ann_factor = 252
    out = pd.DataFrame(index=wide.columns)
    out["cum_return"] = (wide.iloc[-1] / wide.iloc[0] - 1.0).astype(float)
    out["ann_vol"] = (r.std() * np.sqrt(ann_factor)).astype(float)
    out["maxdd"] = wide.apply(mdd).astype(float)
    return out.sort_values("cum_return", ascending=False)

for name, path in SCENARIOS.items():
    panel = load_panel(path)
    wide = panel_to_wide(panel).ffill().bfill()
    print("\n---", name, "---")
    display(scenario_stats(wide).round(4))



--- bear_1y ---


,cum_return,ann_vol,maxdd
asset,,,
IAU,-0.0640,0.2536,-0.2154
EWC,-0.1450,0.3693,-0.3786
EWJ,-0.2470,0.3381,-0.3978
IEMG,-0.2524,0.3937,-0.4836
BTC-USD,-0.2630,1.5770,-0.8685
SPY,-0.2889,0.3695,-0.4522
IEUR,-0.2968,0.3895,-0.4512
ETH-USD,-0.8516,2.1281,-0.9810



--- side_lowvol_1y ---


,cum_return,ann_vol,maxdd
asset,,,
ETH-USD,0.3552,0.7362,-0.5799
IEMG,0.1415,0.1423,-0.0971
EWC,0.0742,0.1246,-0.1037
IAU,0.0289,0.1036,-0.1322
SPY,-0.0059,0.1213,-0.1192
IEUR,-0.0070,0.1282,-0.1380
EWJ,-0.0125,0.1361,-0.1230
BTC-USD,-0.0276,0.5206,-0.4480



--- side_highvol_1y ---


,cum_return,ann_vol,maxdd
asset,,,
BTC-USD,0.8978,1.3774,-0.6993
IAU,0.2003,0.2801,-0.2374
ETH-USD,0.1711,2.3304,-0.9223
EWC,0.0349,0.3020,-0.2450
SPY,0.0104,0.2666,-0.2547
IEUR,-0.0702,0.3201,-0.3512
EWJ,-0.0714,0.2908,-0.3412
IEMG,-0.1673,0.3506,-0.3916
